# Notebook 18 — Submission QA Gate

Run before submitting any paper. Checks are organized into five categories:

- **A. Dataset Integrity** — v2 CSV shape, state count, temperature range, NaN preservation
- **B. Output File Existence** — all key summary CSVs present from NB09–NB16
- **C–G. Numeric Range Checks** — read actual output CSVs; verify model rankings and RMSE/R² bounds
- **H. NB16 Diagnostic Checks** — GBR baseline CI, protocol gap, dummy baseline
- **I. Cross-Manuscript Consistency** — Paper 8 RMSE matches NB16 output; no v1 state count
- **J. Protocol Correctness** — notebooks use v2 path and Pipeline-based imputation
- **K. Structural Checks** — DummyRegressor, PICP metrics, HDBSCAN present in notebooks


In [ ]:
import sys, io, json, re
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
from pathlib import Path
import pandas as pd
import numpy as np

NB_ROOT = Path(r'c:/Users/obalo/Downloads/PhDOneDrive/PhD_Research_Operating_System'
               r'/github_org_bootstrap/phd-geothermal-ml/manual_bootstrap'
               r'/step_by_step_notebooks')
DATA_ROOT = Path(r'c:/Users/obalo/Downloads/PhDOneDrive/PhD_Research_Operating_System/03_data_processed')
PAPER_ROOT = Path(r'c:/Users/obalo/Downloads/PhDOneDrive')

results = []  # {id, category, description, status, actual, expected}

def chk(check_id, category, description, condition, actual, expected, warn_only=False):
    status = 'PASS' if condition else ('WARN' if warn_only else 'FAIL')
    results.append({'id': check_id, 'category': category, 'description': description,
                    'status': status, 'actual': str(actual), 'expected': str(expected)})

print('Paths initialized.')


## A. Dataset Integrity

In [ ]:
# QA-A: v2 canonical dataset integrity
v2 = pd.read_csv(DATA_ROOT / 'geothermal_canonical_v2_0_0.csv')

chk('A01', 'A-Dataset', 'Row count == 2684', len(v2) == 2684, len(v2), 2684)
chk('A02', 'A-Dataset', 'State count == 11', v2['State'].nunique() == 11, v2['State'].nunique(), 11)
chk('A03', 'A-Dataset', 'Temp min >= 90.0 C', v2['Temperature(C)'].min() >= 90.0,
     round(v2['Temperature(C)'].min(), 1), '>= 90.0')
chk('A04', 'A-Dataset', 'Temp max <= 349.0 C', v2['Temperature(C)'].max() <= 349.0,
     round(v2['Temperature(C)'].max(), 1), '<= 349.0')

# NaN must be present — confirms no global pre-imputation
max_nan = v2.drop(columns=['Temperature(C)', 'State'], errors='ignore').isnull().sum().max()
chk('A05', 'A-Dataset', 'NaN preserved (max feature NaN count > 500)', max_nan > 500, max_nan, '> 500')

# Spatial holdout eligibility: at least 6 states with >= 30 samples
eligible = (v2['State'].value_counts() >= 30).sum()
chk('A06', 'A-Dataset', 'At least 6 states eligible for spatial holdout (>= 30 samples)',
     eligible >= 6, eligible, '>= 6')

print('Dataset checks done.')


## B. Output File Existence

In [ ]:
# QA-B: All summary CSVs exist
output_files = [
    ('B01', '09_tree_family_benchmark_no_scripts/outputs/summary/tables/overall_model_ranking.csv'),
    ('B02', '09_tree_family_benchmark_no_scripts/outputs/summary/tables/all_model_protocol_summary.csv'),
    ('B03', '10_tree_family_final_recommendation_no_scripts/outputs/summary/tables/config_weighted_ranking.csv'),
    ('B04', '11_other_ensemble_no_scripts/outputs/summary/tables/overall_model_ranking.csv'),
    ('B05', '11_other_ensemble_no_scripts/outputs/summary/tables/all_model_protocol_summary.csv'),
    ('B06', '12_ablation_feature_families_no_scripts/outputs/summary/tables/protocol_summary_feature_family_ablation.csv'),
    ('B07', '12_ablation_feature_families_no_scripts/outputs/summary/tables/weighted_leaderboard_feature_family_ablation.csv'),
    ('B08', '14_facies_cluster_modeling_no_scripts/outputs/summary/tables/protocol_summary_facies_modeling.csv'),
    ('B09', '14_facies_cluster_modeling_no_scripts/outputs/summary/tables/weighted_leaderboard_facies_modeling.csv'),
    ('B10', '16_validation_diagnostics_no_scripts/outputs/summary/tables/model_protocol_summary_with_ci.csv'),
    ('B11', '16_validation_diagnostics_no_scripts/outputs/summary/tables/robustness_seed_sensitivity_random_split.csv'),
]

for chk_id, rel_path in output_files:
    p = NB_ROOT / rel_path
    chk(chk_id, 'B-Outputs', p.name + ' exists', p.exists(), p.exists(), True)

print('Output file checks done.')


## C. NB09 — Tree Family Benchmark

In [ ]:
# QA-C: NB09 numeric checks
nb09_sum = pd.read_csv(NB_ROOT / '09_tree_family_benchmark_no_scripts/outputs/summary/tables/all_model_protocol_summary.csv')
nb09_rank = pd.read_csv(NB_ROOT / '09_tree_family_benchmark_no_scripts/outputs/summary/tables/overall_model_ranking.csv')

# CatBoost should have the lowest avg_rank (best overall)
top_model = nb09_rank.sort_values('avg_rank').iloc[0]
chk('C01', 'C-NB09', 'CatBoost has lowest weighted rank across protocols',
     top_model['model_name'] == 'catboost',
     f"{top_model['model_name']} ({top_model['avg_rank']:.2f})", 'catboost')

# CatBoost spatial RMSE plausible range
cat_spat = nb09_sum.loc[(nb09_sum['model_name']=='catboost') & (nb09_sum['protocol']=='spatial_holdout'), 'rmse'].values[0]
chk('C02', 'C-NB09', 'CatBoost spatial RMSE in [22, 34] C', 22 <= cat_spat <= 34, round(cat_spat, 2), '[22, 34]')

# XGBoost random RMSE plausible range
xgb_rand = nb09_sum.loc[(nb09_sum['model_name']=='xgboost') & (nb09_sum['protocol']=='random_split'), 'rmse'].values[0]
chk('C03', 'C-NB09', 'XGBoost random RMSE in [15, 22] C', 15 <= xgb_rand <= 22, round(xgb_rand, 2), '[15, 22]')

# Protocol gap positive for all models: spatial RMSE > random RMSE
models = nb09_sum['model_name'].unique()
for model in models:
    rand_rows = nb09_sum[(nb09_sum['model_name']==model) & (nb09_sum['protocol']=='random_split')]['rmse']
    spat_rows = nb09_sum[(nb09_sum['model_name']==model) & (nb09_sum['protocol']=='spatial_holdout')]['rmse']
    if len(rand_rows) > 0 and len(spat_rows) > 0:
        gap = spat_rows.values[0] - rand_rows.values[0]
        chk(f'C04_{model[:5]}', 'C-NB09', f'{model}: spatial > random (gap > 0)', gap > 0,
             round(gap, 2), '> 0')

print('NB09 checks done.')


## D. NB10 — GBR Hyperparameter Tuning

In [ ]:
# QA-D: NB10 checks
nb10_rank = pd.read_csv(NB_ROOT / '10_tree_family_final_recommendation_no_scripts/outputs/summary/tables/config_weighted_ranking.csv')

best_cfg = nb10_rank.sort_values('weighted_rank').iloc[0]
chk('D01', 'D-NB10', 'GBR cfg_1 has lowest weighted rank',
     best_cfg['model_name'] == 'gradient_boosting' and best_cfg['config_id'] == 'cfg_1',
     f"{best_cfg['model_name']} {best_cfg['config_id']} (rank={best_cfg['weighted_rank']:.1f})",
     'gradient_boosting cfg_1')

gbr_cfg1 = nb10_rank[(nb10_rank['model_name']=='gradient_boosting') & (nb10_rank['config_id']=='cfg_1')]
if len(gbr_cfg1) > 0:
    wr = gbr_cfg1.iloc[0]['weighted_rank']
    chk('D02', 'D-NB10', 'GBR cfg_1 weighted rank <= 3.0', wr <= 3.0, round(wr, 1), '<= 3.0')

print('NB10 checks done.')


## E. NB11 — Other Ensembles

In [ ]:
# QA-E: NB11 ensemble checks
nb11_sum = pd.read_csv(NB_ROOT / '11_other_ensemble_no_scripts/outputs/summary/tables/all_model_protocol_summary.csv')
nb11_rank = pd.read_csv(NB_ROOT / '11_other_ensemble_no_scripts/outputs/summary/tables/overall_model_ranking.csv')

# VotingRegressor should have the lowest avg_rank
top_ens = nb11_rank.sort_values('avg_rank').iloc[0]
chk('E01', 'E-NB11', 'VotingRegressor has lowest weighted rank',
     top_ens['model_name'] == 'voting',
     f"{top_ens['model_name']} ({top_ens['avg_rank']:.2f})", 'voting')

# StackingRegressor spatial RMSE > VotingRegressor spatial RMSE (penalization confirmed)
voting_spat = nb11_sum[(nb11_sum['model_name']=='voting') & (nb11_sum['protocol']=='spatial_holdout')]['rmse'].values[0]
stacking_spat = nb11_sum[(nb11_sum['model_name']=='stacking') & (nb11_sum['protocol']=='spatial_holdout')]['rmse'].values[0]
chk('E02', 'E-NB11', 'StackingRegressor spatial RMSE > VotingRegressor spatial (Stacking penalized)',
     stacking_spat > voting_spat,
     f'stacking={round(stacking_spat,2)}, voting={round(voting_spat,2)}', 'stacking > voting')

# VotingRegressor spatial RMSE plausible range
chk('E03', 'E-NB11', 'VotingRegressor spatial RMSE in [22, 35] C', 22 <= voting_spat <= 35,
     round(voting_spat, 2), '[22, 35]')

# Protocol gap for VotingRegressor
voting_rand = nb11_sum[(nb11_sum['model_name']=='voting') & (nb11_sum['protocol']=='random_split')]['rmse'].values[0]
gap = voting_spat - voting_rand
chk('E04', 'E-NB11', 'VotingRegressor protocol gap (spatial - random) > 5 C', gap > 5,
     round(gap, 2), '> 5')

print('NB11 checks done.')


## F. NB12 — Feature Family Ablation

In [ ]:
# QA-F: NB12 ablation checks (VotingRegressor)
nb12_sum = pd.read_csv(NB_ROOT / '12_ablation_feature_families_no_scripts/outputs/summary/tables/protocol_summary_feature_family_ablation.csv')

vot_spat = (nb12_sum[(nb12_sum['model_name']=='voting') & (nb12_sum['protocol']=='spatial_holdout')]
            .set_index('feature_family')['rmse'])

combined = vot_spat.get('combined_all', None)
physical = vot_spat.get('physical_only', None)
chemical = vot_spat.get('chemical_only', None)

if combined is not None and physical is not None:
    chk('F01', 'F-NB12', 'combined_all VotingRegressor spatial < physical_only spatial',
         combined < physical,
         f'comb={round(combined,2)}, phys={round(physical,2)}', 'combined < physical')

if combined is not None and chemical is not None:
    chk('F02', 'F-NB12', 'combined_all VotingRegressor spatial < chemical_only spatial',
         combined < chemical,
         f'comb={round(combined,2)}, chem={round(chemical,2)}', 'combined < chemical')

if chemical is not None and physical is not None:
    chk('F03', 'F-NB12', 'chemical_only spatial >= physical_only spatial (geochemical not best)',
         chemical >= physical,
         f'chem={round(chemical,2)}, phys={round(physical,2)}', 'chemical >= physical')

# All VotingRegressor spatial RMSEs should be < 40
if len(vot_spat) > 0:
    max_spat = vot_spat.max()
    chk('F04', 'F-NB12', 'All VotingRegressor spatial RMSEs < 40 C', max_spat < 40,
         round(max_spat, 2), '< 40')

# combined_all should have lowest spatial RMSE among VotingRegressor families
if len(vot_spat) > 0:
    best_fam = vot_spat.idxmin()
    chk('F05', 'F-NB12', 'combined_all has lowest VotingRegressor spatial RMSE',
         best_fam == 'combined_all', best_fam, 'combined_all')

print('NB12 checks done.')


## G. NB14 — Facies Modeling

In [ ]:
# QA-G: NB14 facies checks
nb14_ld = pd.read_csv(NB_ROOT / '14_facies_cluster_modeling_no_scripts/outputs/summary/tables/weighted_leaderboard_facies_modeling.csv')
nb14_sum = pd.read_csv(NB_ROOT / '14_facies_cluster_modeling_no_scripts/outputs/summary/tables/protocol_summary_facies_modeling.csv')

best_strat = nb14_ld.sort_values('weighted_score').iloc[0]
chk('G01', 'G-NB14', 'local_experts is best strategy (lowest weighted score)',
     best_strat['strategy'] == 'local_experts',
     f"{best_strat['strategy']} ({best_strat['weighted_score']})", 'local_experts')

le_score = nb14_ld.loc[nb14_ld['strategy']=='local_experts', 'weighted_score'].values[0]
gb_score = nb14_ld.loc[nb14_ld['strategy']=='global_baseline', 'weighted_score'].values[0]
chk('G02', 'G-NB14', 'local_experts score < global_baseline score (v2 reversal confirmed)',
     le_score < gb_score,
     f'local={le_score}, global={gb_score}', 'local < global')

le_spat = nb14_sum[(nb14_sum['strategy']=='local_experts') & (nb14_sum['protocol']=='spatial_holdout')]['rmse'].values[0]
gb_spat = nb14_sum[(nb14_sum['strategy']=='global_baseline') & (nb14_sum['protocol']=='spatial_holdout')]['rmse'].values[0]
chk('G03', 'G-NB14', 'local_experts spatial RMSE < global_baseline spatial RMSE',
     le_spat < gb_spat,
     f'local={round(le_spat,2)}, global={round(gb_spat,2)}', 'local < global')

# local_experts spatial RMSE plausible range
chk('G04', 'G-NB14', 'local_experts spatial RMSE in [22, 35] C', 22 <= le_spat <= 35,
     round(le_spat, 2), '[22, 35]')

print('NB14 checks done.')


## H. NB16 — Validation Diagnostics

In [ ]:
# QA-H: NB16 diagnostic checks
nb16 = pd.read_csv(NB_ROOT / '16_validation_diagnostics_no_scripts/outputs/summary/tables/model_protocol_summary_with_ci.csv')

gb_rand = nb16[(nb16['model_name']=='gradient_boosting') & (nb16['protocol']=='random_split')].iloc[0]
gb_spat = nb16[(nb16['model_name']=='gradient_boosting') & (nb16['protocol']=='spatial_holdout')].iloc[0]

chk('H01', 'H-NB16', 'GB random RMSE in [18, 23] C', 18 <= gb_rand['rmse_mean'] <= 23,
     round(gb_rand['rmse_mean'], 2), '[18, 23]')
chk('H02', 'H-NB16', 'GB random R2 > 0.55', gb_rand['r2_mean'] > 0.55,
     round(gb_rand['r2_mean'], 3), '> 0.55')
chk('H03', 'H-NB16', 'GB spatial RMSE > GB random RMSE (protocol gap positive)',
     gb_spat['rmse_mean'] > gb_rand['rmse_mean'],
     f'spatial={round(gb_spat["rmse_mean"],2)} > random={round(gb_rand["rmse_mean"],2)}',
     'spatial > random')

ci_width = gb_rand['rmse_ci95_high'] - gb_rand['rmse_ci95_low']
chk('H04', 'H-NB16', 'GB 95% CI width < 5 C (stable estimation)', ci_width < 5,
     round(ci_width, 2), '< 5')

# DummyRegressor RMSE should be larger than GBR RMSE
dummy_rows = nb16[nb16['model_name'].str.contains('dummy') & (nb16['protocol']=='random_split')]
if len(dummy_rows) > 0:
    dummy_rmse = dummy_rows.iloc[0]['rmse_mean']
    chk('H05', 'H-NB16', 'DummyRegressor RMSE > GBR RMSE (model beats naive baseline)',
         dummy_rmse > gb_rand['rmse_mean'],
         f'dummy={round(dummy_rmse,2)}, gb={round(gb_rand["rmse_mean"],2)}',
         'dummy > gb')

# n_splits for random_split should be 10 (canonical seed count)
chk('H06', 'H-NB16', 'Random split uses 10 seeds (n_splits == 10)',
     gb_rand['n_splits'] == 10, int(gb_rand['n_splits']), 10)

print('NB16 checks done.')


## I. Cross-Manuscript Consistency

In [ ]:
# QA-I: Cross-manuscript consistency
import re

# Paper 8 GB random RMSE should match NB16 output
paper8_text = (PAPER_ROOT / 'Paper8_Journal_Manuscript.md').read_text(encoding='utf-8')
# Pattern: bold GB row in Table 1 e.g. | **Gradient Boosting** | **20.34 ± 1.44** |
match = re.search(r'\*\*Gradient Boosting\*\*.*?\*\*(\d{2}\.\d{2}) ±', paper8_text)
if match:
    p8_rmse = float(match.group(1))
    nb16_rmse = round(gb_rand['rmse_mean'], 2)
    chk('I01', 'I-Consistency',
         f'Paper 8 GB RMSE ({p8_rmse}) matches NB16 output ({nb16_rmse}) within 0.15 C',
         abs(p8_rmse - nb16_rmse) <= 0.15, p8_rmse, f'{nb16_rmse} +/- 0.15')
else:
    results.append({'id': 'I01', 'category': 'I-Consistency',
                    'description': 'Paper 8: GB RMSE table entry found',
                    'status': 'WARN', 'actual': 'regex not matched', 'expected': 'bold GB row'})

# No v1 state count in any manuscript
docs = [(f'Paper{i}', f'Paper{i}_Journal_Manuscript.md') for i in range(1, 10)]
docs.append(('Dissertation', 'Dissertation_Manuscript.md'))
for doc_name, fn in docs:
    txt = (PAPER_ROOT / fn).read_text(encoding='utf-8')
    has_v1 = ('17 US states' in txt) or ('17 U.S. states' in txt)
    chk(f'I02_{doc_name[:5]}', 'I-Consistency',
         f'{doc_name}: no v1 state count ("17 states")',
         not has_v1, has_v1, False)

# Paper 4 should mention 'combined_all' or 'combined-all' (deployment champion change)
paper4_text = (PAPER_ROOT / 'Paper4_Journal_Manuscript.md').read_text(encoding='utf-8')
has_combined = 'combined' in paper4_text.lower()
chk('I03', 'I-Consistency', 'Paper 4 references combined-all feature family', has_combined, has_combined, True)

print('Consistency checks done.')


## J. Protocol Correctness

In [ ]:
# QA-J: Protocol correctness — v2 path + Pipeline-based imputation
nb_protocol_checks = [
    ('J01', '09_tree_family_benchmark_no_scripts/09_tree_family_benchmark.ipynb',
     'geothermal_canonical_v2', 'v2 dataset path'),
    ('J02', '11_other_ensemble_no_scripts/11_other_ensemble.ipynb',
     'geothermal_canonical_v2', 'v2 dataset path'),
    ('J03', '12_ablation_feature_families_no_scripts/12_ablation_feature_families.ipynb',
     'geothermal_canonical_v2', 'v2 dataset path'),
    ('J04', '16_validation_diagnostics_no_scripts/16_validation_diagnostics.ipynb',
     'geothermal_canonical_v2', 'v2 dataset path'),
    ('J05', '09_tree_family_benchmark_no_scripts/09_tree_family_benchmark.ipynb',
     'Pipeline', 'Pipeline-based imputation (per-split)'),
    ('J06', '11_other_ensemble_no_scripts/11_other_ensemble.ipynb',
     'Pipeline', 'Pipeline-based imputation (per-split)'),
    ('J07', '12_ablation_feature_families_no_scripts/12_ablation_feature_families.ipynb',
     'Pipeline', 'Pipeline-based imputation (per-split)'),
]

for chk_id, nb_rel, search_str, desc in nb_protocol_checks:
    p = NB_ROOT / nb_rel
    if p.exists():
        content = p.read_text(encoding='utf-8', errors='ignore')
        found = search_str in content
        chk(chk_id, 'J-Protocol', f'{p.name}: {desc}', found, found, True)
    else:
        results.append({'id': chk_id, 'category': 'J-Protocol',
                        'description': f'{nb_rel} not found',
                        'status': 'FAIL', 'actual': 'File not found', 'expected': 'File exists'})

print('Protocol correctness checks done.')


## K. Structural Checks

In [ ]:
# QA-K: Structural checks — key algorithms/metrics present in notebooks
struct_checks = [
    ('K01', '09_tree_family_benchmark_no_scripts/09_tree_family_benchmark.ipynb',
     'DummyRegressor', 'DummyRegressor baseline present'),
    ('K02', '13_uncertainty_conformal_no_scripts/13_uncertainty_conformal.ipynb',
     'PICP', 'PICP conformal metric present'),
    ('K03', '13_uncertainty_conformal_no_scripts/13_uncertainty_conformal.ipynb',
     'ACE', 'ACE conformal metric present'),
    ('K04', '14_facies_cluster_modeling_no_scripts/14_facies_cluster_modeling.ipynb',
     'HDBSCAN', 'HDBSCAN facies clustering present'),
    ('K05', '16_validation_diagnostics_no_scripts/16_validation_diagnostics.ipynb',
     'json', 'JSON model serialization present'),
    ('K06', '11_other_ensemble_no_scripts/11_other_ensemble.ipynb',
     'VotingRegressor', 'VotingRegressor ensemble present'),
    ('K07', '11_other_ensemble_no_scripts/11_other_ensemble.ipynb',
     'StackingRegressor', 'StackingRegressor ensemble present'),
]

for chk_id, nb_rel, search_str, desc in struct_checks:
    p = NB_ROOT / nb_rel
    if p.exists():
        content = p.read_text(encoding='utf-8', errors='ignore')
        found = search_str in content
        chk(chk_id, 'K-Structural', f'{p.name}: {desc}', found, found, True)
    else:
        results.append({'id': chk_id, 'category': 'K-Structural',
                        'description': f'{nb_rel} not found',
                        'status': 'FAIL', 'actual': 'File not found', 'expected': 'File exists'})

print('Structural checks done.')


## Summary & Gate Decision

In [ ]:
# Final QA gate summary
qa_df = pd.DataFrame(results)
passed = (qa_df['status'] == 'PASS').sum()
warned = (qa_df['status'] == 'WARN').sum()
failed = (qa_df['status'] == 'FAIL').sum()

print('=' * 70)
print(f'SUBMISSION QA GATE: {passed} PASS | {warned} WARN | {failed} FAIL  (total {len(qa_df)})')
print('=' * 70)

for cat in qa_df['category'].unique():
    sub = qa_df[qa_df['category'] == cat]
    sub_pass = (sub['status'] == 'PASS').sum()
    print(f'\n--- {cat} ({sub_pass}/{len(sub)}) ---')
    for _, row in sub.iterrows():
        marker = 'PASS' if row['status'] == 'PASS' else row['status']
        print(f"  [{marker}] {row['id']}: {row['description']}")
        if row['status'] != 'PASS':
            print(f"         actual={row['actual']}  expected={row['expected']}")

# Display as DataFrame
display(qa_df[qa_df['status'] != 'PASS'][['id','category','description','status','actual','expected']])

if failed > 0:
    print(f'\n>>> {failed} CHECKS FAILED -- DO NOT SUBMIT until resolved <<<')
    raise SystemExit(f'{failed} QA check(s) failed')
elif warned > 0:
    print(f'\n>>> ALL PASS; {warned} warnings -- review before submitting <<<')
else:
    print('\n>>> ALL CHECKS PASSED -- Ready for submission review <<<')
